# Yelp GNN — temporal-burst co-edges (corrected graph, Colab GPU)

Two changes vs the previous co-edge run:
1. **Corrected graph** — `build_yelp_graph_full.py` now reads the right YelpChi reviewer column (`p[2]`, not the review-id `p[1]`), so ~28k bogus singleton reviewers collapse into real reviewers (487,897 → 459,492). Chi finally contributes real coordination structure.
2. **Temporal-burst co-edges** — coordinated crews hit the same targets in *bursts*; genuine co-reviewers are spread over years. `reviewer_edges_temporal.npz` links two reviewers only if they share ≥2 businesses **and** co-reviewed ≥1 of them within **30 days** (12,277 edges, vs 96,904 plain).

Trains **three models, identical seed + loss (CE inverse-frequency)** so only the graph differs:
- **BASE** — 4 relations, no co-edges.
- **CO** — + plain co-edges (`reviewer_edges.npz`).
- **CO-temporal** — + temporal-burst co-edges (`reviewer_edges_temporal.npz`).

**Setup:** GPU runtime. Upload `graph_full.npz`, `reviewer_edges.npz`, `reviewer_edges_temporal.npz` (all from `data/yelp_graph/`). Run all → `yelp_gnn_temporal_preds.npz`.

In [ ]:
!pip -q install torch_geometric
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
import numpy as np, torch
from torch_geometric.data import HeteroData
d   = np.load('graph_full.npz')
co  = np.load('reviewer_edges.npz')['edge_index']
cot = np.load('reviewer_edges_temporal.npz')['edge_index']
def zscore(a):
    a = torch.tensor(a, dtype=torch.float); m, s = a.mean(0, keepdim=True), a.std(0, keepdim=True)
    return (a - m) / (s + 1e-6)
data = HeteroData()
data['review'].x   = zscore(d['review_feat'])
data['reviewer'].x = zscore(d['reviewer_feat'])
data['business'].x = zscore(d['business_feat'])
rid = torch.arange(d['review_feat'].shape[0])
ru  = torch.tensor(d['review_user'], dtype=torch.long); rb = torch.tensor(d['review_biz'], dtype=torch.long)
data['review','by','reviewer'].edge_index    = torch.stack([rid, ru])
data['reviewer','wrote','review'].edge_index  = torch.stack([ru, rid])
data['review','on','business'].edge_index     = torch.stack([rid, rb])
data['business','has','review'].edge_index     = torch.stack([rb, rid])
data['review'].y = torch.tensor(d['review_label'], dtype=torch.long)
split = torch.tensor(d['review_split'], dtype=torch.long)
data['review'].train_mask = (split==1); data['review'].val_mask=(split==2); data['review'].test_mask=(split==3)
CO_PLAIN = torch.tensor(co,  dtype=torch.long)
CO_TEMP  = torch.tensor(cot, dtype=torch.long)
print(data)
print('fake rate', float((data['review'].y==1).float().mean()), '| plain co', co.shape[1], '| temporal co', cot.shape[1])

In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, SAGEConv, Linear
REL_BASE = [('review','by','reviewer'),('reviewer','wrote','review'),('review','on','business'),('business','has','review')]
REL_CO   = REL_BASE + [('reviewer','co','reviewer')]
class HeteroGNN(torch.nn.Module):
    def __init__(self, rels, hidden=128, layers=2, out=2):
        super().__init__()
        self.proj = torch.nn.ModuleDict({nt: Linear(-1, hidden) for nt in ['review','reviewer','business']})
        self.convs = torch.nn.ModuleList([HeteroConv({r: SAGEConv((-1,-1), hidden) for r in rels}, aggr='sum') for _ in range(layers)])
        self.dropout = torch.nn.Dropout(0.3); self.out = Linear(hidden, out)
    def forward(self, x, e):
        x = {k: self.proj[k](v).relu() for k,v in x.items()}
        for c in self.convs: x = {k: self.dropout(v.relu()) for k,v in c(x,e).items()}
        return self.out(x['review'])

In [ ]:
from sklearn.metrics import f1_score, recall_score, precision_score
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); data = data.to(dev)
CO_PLAIN = CO_PLAIN.to(dev); CO_TEMP = CO_TEMP.to(dev)
tr,va,te = data['review'].train_mask, data['review'].val_mask, data['review'].test_mask
yb = data['review'].y; ytr = yb[tr].cpu().numpy(); n0=(ytr==0).sum(); n1=(ytr==1).sum()
W = torch.tensor([len(ytr)/(2*n0), len(ytr)/(2*n1)], dtype=torch.float, device=dev)  # CE inverse-frequency
base_edict = {k:v for k,v in data.edge_index_dict.items() if k[1] != 'co'}
def edict_with(co_edges):
    e = dict(base_edict); e[('reviewer','co','reviewer')] = co_edges; return e
def evaluate(model, edict, mask):
    model.eval()
    with torch.no_grad(): pr = model(data.x_dict, edict).argmax(1)
    yt=yb[mask].cpu().numpy(); yp=pr[mask].cpu().numpy()
    return f1_score(yt,yp,average='macro',zero_division=0), recall_score(yt,yp,pos_label=1,zero_division=0), precision_score(yt,yp,pos_label=1,zero_division=0)
def run(rels, edict):
    torch.manual_seed(0)
    model = HeteroGNN(rels).to(dev)
    with torch.no_grad(): model(data.x_dict, edict)
    opt = torch.optim.Adam(model.parameters(), lr=5e-3, weight_decay=5e-4)
    best, state, bad = -1, None, 0
    for ep in range(1,151):
        model.train(); opt.zero_grad(); out = model(data.x_dict, edict)
        loss = F.cross_entropy(out[tr], yb[tr], weight=W)
        loss.backward(); opt.step()
        v = evaluate(model, edict, va)[0]
        if v>best: best, state, bad = v, {k:vv.cpu().clone() for k,vv in model.state_dict().items()}, 0
        else: bad += 1
        if bad>=15: break
    model.load_state_dict(state)
    with torch.no_grad(): prob = torch.softmax(model(data.x_dict, edict),1)[:,1].cpu().numpy()
    mf1,fr,fp = evaluate(model, edict, te)
    return prob, mf1, fr, fp
saved = {'review_user': d['review_user'], 'review_biz': d['review_biz'], 'label': d['review_label'], 'split': d['review_split']}
VARIANTS = [('BASE (no co-edges)', REL_BASE, base_edict, 'prob_base'),
            ('CO (plain)',         REL_CO,   edict_with(CO_PLAIN), 'prob_co'),
            ('CO-temporal',        REL_CO,   edict_with(CO_TEMP),  'prob_co_temporal')]
print(f"{'variant':24}{'testMF1':>9}{'fakeR':>7}{'fakeP':>7}{'pred-fake%':>11}  (true 12.2%)")
for name, rels, edict, key in VARIANTS:
    prob, mf1, fr, fp = run(rels, edict)
    saved[key] = prob
    print(f"{name:24}{mf1:>8.1%}{fr:>7.1%}{fp:>7.1%}{float((prob>=0.5).mean()):>10.1%}")
np.savez_compressed('yelp_gnn_temporal_preds.npz', **saved)
print('\nsaved yelp_gnn_temporal_preds.npz (prob_base + prob_co + prob_co_temporal + structure + labels)')